# Transform pipeline validation: eye-in-hand ball detector

Validates `abb_irb140_perception/abb_irb140_perception_node.py`'s
`depth_camera_optical -> base_link` TF chain against ground truth, using:

- **Ground truth**: `notebooks/sphere_pose_ground_truth.txt`, captured via
  `gz model -m my_sphere -p` (sphere position/orientation in the **world**
  frame).
- **Bag**: `notebooks/perception_inputs/` (mcap) — RGB, depth, and
  `camera_info` only. **No `/tf` or `/joint_states` were recorded.**

## Key assumption (please confirm)

Because the bag has no TF/joint data, this notebook assumes the arm was
**stationary at the sim's startup joint configuration** for the whole
recording — i.e. the `initial_value` block in
`abb_irb140_description/urdf/irb140.xacro` (ros2_control `GazeboSimSystem`
seed values), **not** the MoveIt SRDF `home_pose` (those differ: e.g.
joint_2 is 0.0873 rad at sim start vs 0.5632 rad in `home_pose`).

This assumption is corroborated below: forward-kinematics through the full
URDF chain at the sim-start joint config predicts the `depth_camera` offset
from `base_link` as **x=0.4339 m, z=0.5191 m**, matching your manual
measurement (x=0.434, z=0.519) to within a millimetre. If the arm actually
moved during the bag, this notebook's transform will be wrong for later
frames — re-record with `/tf` (or at least `/joint_states`) to lift this
assumption.

## What this notebook checks, and why split this way

1. **Test 1 — pure geometry, no detector.** Project the ground-truth sphere
   pose through `world -> base_link -> depth_camera_optical -> pixel` and
   compare the predicted pixel/depth against the bag's actual depth image.
   This isolates the **static TF chain** (URDF offsets, rotation
   conventions, spawn pose) from any detector error.
2. **Test 2 — full pipeline.** Run the same YOLO + back-projection code the
   node uses on a real bag frame, transform the result to `base_link` and
   `world`, and compare to ground truth. If Test 1 passes but Test 2 fails,
   the bug is in detection/back-projection, not the transform chain.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

import rosbag2_py
from rosidl_runtime_py.utilities import get_message
from rclpy.serialization import deserialize_message

from image_geometry import PinholeCameraModel
from cv_bridge import CvBridge

from hybraut_irb140.perception_geometry import backproject_pixel, depth_to_meters

NOTEBOOK_DIR = Path.cwd()
BAG_DIR = NOTEBOOK_DIR / "perception_inputs"
GT_FILE = NOTEBOOK_DIR / "sphere_pose_ground_truth.txt"

bridge = CvBridge()


## 1. Ground truth and rigid, known-exact constants

- Sphere pose: parsed from `gz model -p` output (world frame).
- `world -> base_link`: the spawn pose from
  `abb_irb140_description/launch/sim_robot.launch.py`
  (`-x 0.02 -y 0.0 -z 0.97`, **no** `-R/-P/-Y` given, so identity rotation).
  Matches what you measured.
- Sphere radius: `abb_irb140_description/worlds/robot_lab.world`,
  `sphere_collision` -> `<radius>0.02</radius>`. Used because the depth
  camera sees the ball's near *surface*, not its center.

In [ ]:
import re

def parse_gz_model_pose(path):
    """Parses `gz model -m <name> -p` output:
        - Pose [ XYZ (m) ] [ RPY (rad) ]:
            [0.705000 0.014790 1.086900]
            [-0.739514 0.000000 -0.000000]
    """
    text = path.read_text()
    # Purely-numeric bracketed groups only (skips "[robot_lab]", "[12]" labels,
    # and the "[ XYZ (m) ]" / "[ RPY (rad) ]" header, which contain letters).
    groups = re.findall(r"\[\s*(-?\d[-\d.\s]*)\]", text)
    numeric_groups = [g for g in groups if len(g.split()) == 3]
    if len(numeric_groups) != 2:
        raise ValueError(f"Expected exactly 2 three-number groups (XYZ, RPY), found {len(numeric_groups)}: {numeric_groups}")
    xyz = np.array([float(x) for x in numeric_groups[0].split()])
    rpy = np.array([float(x) for x in numeric_groups[1].split()])
    return xyz, rpy

GT_SPHERE_XYZ, GT_SPHERE_RPY = parse_gz_model_pose(GT_FILE)
print("Ground truth sphere position (world):", GT_SPHERE_XYZ)
print("Ground truth sphere rpy (world):     ", GT_SPHERE_RPY, "(irrelevant for a sphere's position)")

SPHERE_RADIUS_M = 0.02

WORLD_T_BASE = np.eye(4)
WORLD_T_BASE[:3, 3] = [0.02, 0.0, 0.97]


## 2. Forward kinematics: `base_link -> depth_camera_optical`

Every joint origin/axis below is copied verbatim from the URDF/xacro so
this stays in sync with the robot description:

- `abb_irb140_description/urdf/irb140.xacro` — `joint_1`..`joint_6`,
  `joint_6-tool0`
- `abb_irb140_description/urdf/irb140_pneumatic_gripper.xacro` — the
  gripper mount and the `depth_camera` / `depth_camera_optical` joints
  (called from `irb140.xacro` with `xyz="0.00 0.0 0.08" rpy="0 0 3.142"`)

The `depth_camera -> depth_camera_optical` joint's `rpy` is the standard
ROS optical-frame rotation (Z-forward body axis -> Z-forward/X-right/Y-down
optical axes); see the long comment in the xacro for why `depth_camera`
itself must stay body-convention (+X forward) instead.

In [ ]:
def Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rpy_matrix(r, p, y):
    # ROS/URDF convention: R = Rz(yaw) @ Ry(pitch) @ Rx(roll)
    return Rz(y) @ Ry(p) @ Rx(r)

def homogeneous(xyz, rpy):
    M = np.eye(4)
    M[:3, :3] = rpy_matrix(*rpy)
    M[:3, 3] = xyz
    return M

_AXIS_ROT = {"x": Rx, "y": Ry, "z": Rz}

# (joint_name, parent, child, origin_xyz, origin_rpy, revolute_axis_or_None)
URDF_CHAIN = [
    ("joint_1", "base_link", "link_1", (0, 0, 0), (0, 0, 0), "z"),
    ("joint_2", "link_1", "link_2", (0.058333, 0, 0.2875), (0, 0, 0), "y"),
    ("joint_3", "link_2", "link_3", (0, 0, 0.295833), (0, 0, 0), "y"),
    ("joint_4", "link_3", "link_4", (0, 0, 0), (0, 0, 0), "x"),
    ("joint_5", "link_4", "link_5", (0.310417, 0, 0), (0, 0, 0), "y"),
    ("joint_6", "link_5", "link_6", (0, 0, 0.004167), (0, 0, 0), "x"),
    ("joint_6-tool0", "link_6", "tool0", (0, 0, 0), (0, 1.57079632679, 0), None),
    ("pneumatic_gripper_to_tool0", "tool0", "pneumatic_gripper_base_link",
     (0.00, 0.0, 0.08), (0, 0, 3.142), None),
    ("pneumatic_gripper_to_depth_camera", "pneumatic_gripper_base_link", "depth_camera",
     (-0.025, 0.0, 0.05), (3.14159265359, -1.57079632679, 0), None),
    ("depth_camera_to_optical", "depth_camera", "depth_camera_optical",
     (0, 0, 0), (-1.57079632679, 0, -1.57079632679), None),
]

def forward_kinematics(joint_positions):
    """joint_positions: dict of revolute joint name -> angle (rad).
    Returns {link_name: 4x4 transform base_link -> link_name}."""
    frames = {"base_link": np.eye(4)}
    for name, parent, child, xyz, rpy, axis in URDF_CHAIN:
        M = homogeneous(xyz, rpy)
        if axis is not None:
            theta = joint_positions.get(name, 0.0)
            Mrot = np.eye(4)
            Mrot[:3, :3] = _AXIS_ROT[axis](theta)
            M = M @ Mrot
        frames[child] = frames[parent] @ M
    return frames

# Sim-startup joint config: ros2_control `initial_value` state-interface
# seed in irb140.xacro (GazeboSimSystem applies this before any controller
# runs) -- NOT the SRDF `home_pose` group state, which is a different pose.
DEFAULT_JOINT_POSITIONS = {
    "joint_1": 0.0,
    "joint_2": 0.0873,
    "joint_3": -0.283605,
    "joint_4": 0.0,
    "joint_5": 1.256640,
    "joint_6": 0.0,
}

frames = forward_kinematics(DEFAULT_JOINT_POSITIONS)
BASE_T_CAM_OPTICAL = frames["depth_camera_optical"]

print("base_link -> depth_camera_optical translation:", BASE_T_CAM_OPTICAL[:3, 3])
print("Your manual measurement was: x=0.434, z=0.519  -> ", end="")
print("MATCH" if np.allclose(BASE_T_CAM_OPTICAL[:3, 3], [0.434, 0, 0.519], atol=2e-3) else "MISMATCH -- check joint assumption")


## 3. Small geometry helpers

In [ ]:
def invert_rigid(T):
    R, t = T[:3, :3], T[:3, 3]
    Tinv = np.eye(4)
    Tinv[:3, :3] = R.T
    Tinv[:3, 3] = -R.T @ t
    return Tinv

def apply_transform(T, p):
    return T[:3, :3] @ np.asarray(p, dtype=float) + T[:3, 3]

BASE_T_WORLD = invert_rigid(WORLD_T_BASE)
OPTICAL_T_BASE = invert_rigid(BASE_T_CAM_OPTICAL)


## 4. Load one synchronized RGB / depth / camera_info frame from the bag

Bag has no `/tf`, so any frame works under the static-pose assumption; the
bag midpoint is used to avoid startup transients.

In [ ]:
def open_reader(path):
    storage_options = rosbag2_py.StorageOptions(uri=str(path), storage_id="mcap")
    converter_options = rosbag2_py.ConverterOptions("", "")
    reader = rosbag2_py.SequentialReader()
    reader.open(storage_options, converter_options)
    return reader

def bag_target_timestamp_ns(path):
    meta = yaml.safe_load((path / "metadata.yaml").read_text())["rosbag2_bagfile_information"]
    start = meta["starting_time"]["nanoseconds_since_epoch"]
    duration = meta["duration"]["nanoseconds"]
    return start + duration // 2

def read_closest_messages(path, topics, target_ns):
    """Returns {topic: (t_ns, msg)} for the message on each topic closest to target_ns.
    Deserializes only the winning message per topic (bag has ~1500 msgs/topic)."""
    reader = open_reader(path)
    type_map = {t.name: t.type for t in reader.get_all_topics_and_types()}
    reader.set_filter(rosbag2_py.StorageFilter(topics=topics))
    best_raw = {t: None for t in topics}
    while reader.has_next():
        topic, data, t_ns = reader.read_next()
        cur = best_raw[topic]
        if cur is None or abs(t_ns - target_ns) < abs(cur[0] - target_ns):
            best_raw[topic] = (t_ns, data)
    out = {}
    for topic, (t_ns, data) in best_raw.items():
        msg_type = get_message(type_map[topic])
        out[topic] = (t_ns, deserialize_message(data, msg_type))
    return out

TOPICS = ["/camera/color/image_raw", "/camera/depth/image_rect_raw", "/camera/color/camera_info"]
target_ns = bag_target_timestamp_ns(BAG_DIR)
frame = read_closest_messages(BAG_DIR, TOPICS, target_ns)

for topic, (t_ns, msg) in frame.items():
    print(f"{topic}: t={t_ns} ns, dt from target={(t_ns - target_ns) / 1e9:+.3f}s")

rgb_msg = frame["/camera/color/image_raw"][1]
depth_msg = frame["/camera/depth/image_rect_raw"][1]
info_msg = frame["/camera/color/camera_info"][1]

bgr = bridge.imgmsg_to_cv2(rgb_msg, desired_encoding="bgr8")
depth = depth_to_meters(bridge.imgmsg_to_cv2(depth_msg, desired_encoding="passthrough"))

camera_model = PinholeCameraModel()
camera_model.fromCameraInfo(info_msg)
print("\nImage size:", bgr.shape[1], "x", bgr.shape[0])
print("fx, fy, cx, cy:", camera_model.fx(), camera_model.fy(), camera_model.cx(), camera_model.cy())


## 5. Test 1 — pure transform chain (no detector)

Project ground truth `world -> base_link -> depth_camera_optical -> pixel`
and compare against the bag's actual depth image at that pixel. If this is
close, the static TF chain (URDF offsets + rotation conventions + spawn
pose + joint-pose assumption) is correct, independent of detection.

In [ ]:
p_base_gt = apply_transform(BASE_T_WORLD, GT_SPHERE_XYZ)
p_optical_gt = apply_transform(OPTICAL_T_BASE, p_base_gt)

print("Ground truth sphere center in base_link: ", p_base_gt)
print("Ground truth sphere center in depth_camera_optical:", p_optical_gt)

u_gt, v_gt = camera_model.project3dToPixel(tuple(p_optical_gt))
expected_center_depth = p_optical_gt[2]
expected_surface_depth = expected_center_depth - SPHERE_RADIUS_M

h, w = depth.shape[:2]
print(f"\nPredicted pixel: u={u_gt:.1f}, v={v_gt:.1f}  (image is {w}x{h})")

if 0 <= int(round(u_gt)) < w and 0 <= int(round(v_gt)) < h:
    actual_depth_at_pixel = float(depth[int(round(v_gt)), int(round(u_gt))])
    print(f"Expected center depth:  {expected_center_depth:.4f} m")
    print(f"Expected surface depth: {expected_surface_depth:.4f} m  (center - sphere radius)")
    print(f"Actual depth in bag at that pixel: {actual_depth_at_pixel:.4f} m")
    print(f"Surface-depth residual: {actual_depth_at_pixel - expected_surface_depth:+.4f} m")
else:
    print("Predicted pixel falls OUTSIDE the image -- static transform chain is likely wrong "
          "(check the joint-pose assumption, spawn pose, or a sign/axis error in the FK chain above).")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(bgr[:, :, ::-1])
axes[0].scatter([u_gt], [v_gt], c="lime", marker="+", s=200, linewidths=2)
axes[0].set_title("RGB: predicted ball pixel from ground truth")
im = axes[1].imshow(depth, cmap="viridis")
axes[1].scatter([u_gt], [v_gt], c="red", marker="+", s=200, linewidths=2)
axes[1].set_title("Depth (m): predicted ball pixel from ground truth")
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()


**Reading this test:** the `+` marker should land visibly on the sphere in
both images, and the depth residual should be small (within a few cm —
render/geometry noise, not a systematic offset). A marker far from the
ball, or a residual of many centimetres, points at the static transform
chain (Section 2) rather than the detector.

## 6. Test 2 — full pipeline (YOLO + back-projection), same code as the node

Reuses `hybraut_irb140.perception_geometry.backproject_pixel` (identical
function the node imports) and mirrors `abb_irb140_perception_node.py`'s
`_process()`: bbox center -> back-project -> push depth back by one radius
along optical Z -> transform to `base_link`.

The node's default detector config (no `model_path` param, no packaged
`ball_yolo.pt` found under `hybraut_irb140`'s share dir) falls back to
plain `yolov8n.pt`, target classes `{"sports ball", "ball"}`, confidence
0.15 — reproduced here. A generic COCO-trained YOLO may or may not
reliably find a synthetic Gazebo sphere; if it misses, a manual bbox
override is provided below so you can still validate the geometry.

In [ ]:
from ultralytics import YOLO

YOLO_WEIGHTS = "/home/ryan/ros2_ws/yolov8n.pt"
CONF_THRESHOLD = 0.15
TARGET_CLASSES = {"sports ball", "ball"}

model = YOLO(YOLO_WEIGHTS)
result = model.predict(bgr, conf=CONF_THRESHOLD, iou=0.45, imgsz=640, device="cpu", verbose=False)[0]
class_names = model.names

candidates = []
for box in result.boxes:
    name = class_names.get(int(box.cls[0]), str(int(box.cls[0])))
    if TARGET_CLASSES and name not in TARGET_CLASSES:
        continue
    score = float(box.conf[0])
    x1, y1, x2, y2 = (float(v) for v in box.xyxy[0].tolist())
    candidates.append((x1, y1, x2, y2, score, name))

print(f"{len(candidates)} target-class detection(s) at conf>={CONF_THRESHOLD}")
for c in candidates:
    print(" ", c)

# --- Manual override -------------------------------------------------
# If YOLO found nothing (or the wrong thing), inspect the RGB image above,
# eyeball the ball's bounding box in pixels, and set MANUAL_BBOX =
# (x1, y1, x2, y2). Leave as None to use the best YOLO candidate.
MANUAL_BBOX = None

if MANUAL_BBOX is not None:
    x1, y1, x2, y2 = MANUAL_BBOX
    score, name = 1.0, "manual"
elif candidates:
    x1, y1, x2, y2, score, name = max(candidates, key=lambda c: c[4])
else:
    raise RuntimeError("No detection and no MANUAL_BBOX set -- inspect the RGB image and set MANUAL_BBOX.")

u = 0.5 * (x1 + x2)
v = 0.5 * (y1 + y2)
r_px = 0.25 * ((x2 - x1) + (y2 - y1))
print(f"\nUsing bbox ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f}) score={score:.2f} name={name!r}")
print(f"center u={u:.1f} v={v:.1f}  r_px={r_px:.1f}")


In [ ]:
pt_cam = backproject_pixel(
    u, v, depth, camera_model,
    window=2, min_valid_depth_m=0.05,
)
if pt_cam is None:
    raise RuntimeError("backproject_pixel returned None -- no valid depth in the window around (u, v).")

fx = camera_model.fx()
z = pt_cam.point.z
r_m = r_px * z / fx
pt_cam.point.z += r_m  # front-surface depth -> approximate sphere center, same as the node

p_optical_meas = np.array([pt_cam.point.x, pt_cam.point.y, pt_cam.point.z])
p_base_meas = apply_transform(BASE_T_CAM_OPTICAL, p_optical_meas)
p_world_meas = apply_transform(WORLD_T_BASE, p_base_meas)

print("Measured sphere center in depth_camera_optical:", p_optical_meas)
print("Measured sphere center in base_link:            ", p_base_meas)
print("Measured sphere center in world:                ", p_world_meas)

error = p_world_meas - GT_SPHERE_XYZ
print(f"\nGround truth (world): {GT_SPHERE_XYZ}")
print(f"Per-axis error (world): {error}")
print(f"Euclidean error: {np.linalg.norm(error) * 100:.1f} cm")

TOLERANCE_M = 0.05
verdict = "PASS" if np.linalg.norm(error) < TOLERANCE_M else "FAIL"
print(f"\n[{verdict}] against {TOLERANCE_M*100:.0f} cm tolerance")

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(bgr[:, :, ::-1])
ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
ax.scatter([u], [v], c="lime", marker="+", s=150)
ax.scatter([u_gt], [v_gt], c="red", marker="x", s=150, label="ground-truth pixel (Test 1)")
ax.legend()
ax.set_title(f"Test 2 detection vs. Test 1 ground-truth pixel  ({verdict}, err={np.linalg.norm(error)*100:.1f} cm)")
plt.show()


## 8. Multi-position coverage — more comprehensive without a new rosbag

Section 5-6 above validate one detection at one sphere position. The bag
disk (`perception_inputs/`) is 3.2 GB already and this machine was at
**97% disk usage** when this section was added — recording several more
full rosbags at different ball positions was not an option.

Instead, `perception_inputs_multi/pos1..pos4/` hold four **single-frame**
captures (RGB `.png` + compressed depth `.npz` + `camera_info.yaml`, a few
hundred KB each instead of hundreds of MB) grabbed directly from the live
sim by teleporting `my_sphere` via the same `/world/robot_lab/set_pose`
gz-transport call `sim_reset_service.py`'s `/respawn_ball` service uses,
then subscribing for exactly one synced frame. **The arm was never
touched**, so the static-joint-pose assumption from Section 2 still holds
for all of these — this augmentation adds *spatial* coverage (does the
pipeline hold up across the frame, not just at the one pixel the original
bag happened to show), not *joint-configuration* coverage.

| position | base_link (x, y) | pixel region |
|---|---|---|
| pos0 | ~(0.68, 0.01) — original bag | near-center |
| pos1 | (0.55, 0.15) | near, left |
| pos2 | (0.55, -0.15) | near, right |
| pos3 | (0.85, 0.15) | far, left |
| pos4 | (0.85, -0.15) | far, right |

Candidates were chosen by projecting through the *same* FK + camera model
from Section 2-3 first (before touching the sim) to confirm each one lands
comfortably inside the 640x480 frame — so this itself is a small
additional use of the transform chain, independent of the bag data.

In [ ]:
import cv2
from sensor_msgs.msg import CameraInfo
import pandas as pd

def load_lightweight_capture(dir_path):
    """Loads one perception_inputs_multi/posN/ capture (see Section 8)."""
    dir_path = Path(dir_path)
    bgr_ = cv2.imread(str(dir_path / "rgb.png"))
    depth_raw = np.load(dir_path / "depth.npz")["depth"]
    depth_ = depth_to_meters(depth_raw)

    info = yaml.safe_load((dir_path / "camera_info.yaml").read_text())
    info_msg = CameraInfo()
    info_msg.width = info["width"]
    info_msg.height = info["height"]
    info_msg.k = info["k"]
    info_msg.d = info["d"]
    info_msg.distortion_model = info["distortion_model"]
    # image_geometry reads intrinsics from P (post-rectification), not K --
    # the capture script only saved K. This sim camera is undistorted
    # (D all-zero) and monocular (no stereo baseline), so P is just K with
    # an appended zero column and R is identity; confirmed against the
    # original bag's real camera_info, which has exactly this pattern.
    fx_, _, cx_, _, fy_, cy_, *_ = info["k"]
    info_msg.p = [fx_, 0.0, cx_, 0.0, 0.0, fy_, cy_, 0.0, 0.0, 0.0, 1.0, 0.0]
    info_msg.r = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
    cm_ = PinholeCameraModel()
    cm_.fromCameraInfo(info_msg)

    gt_xyz_, _ = parse_gz_model_pose(dir_path / "sphere_pose_ground_truth.txt")
    return bgr_, depth_, cm_, gt_xyz_


In [ ]:
def test1_metrics(depth_, camera_model_, gt_xyz_):
    """Pure-geometry check (Section 5, generalized): project ground truth to
    a pixel/expected depth and compare against the actual depth image."""
    p_base_ = apply_transform(BASE_T_WORLD, gt_xyz_)
    p_opt_ = apply_transform(OPTICAL_T_BASE, p_base_)
    u_, v_ = camera_model_.project3dToPixel(tuple(p_opt_))
    h_, w_ = depth_.shape[:2]
    inside_ = 0 <= int(round(u_)) < w_ and 0 <= int(round(v_)) < h_
    expected_surface_ = p_opt_[2] - SPHERE_RADIUS_M
    if inside_:
        actual_ = float(depth_[int(round(v_)), int(round(u_))])
        residual_ = actual_ - expected_surface_
    else:
        actual_ = residual_ = float("nan")
    return dict(u=u_, v=v_, inside=inside_,
                expected_surface_depth=expected_surface_,
                actual_depth=actual_, depth_residual=residual_)


def test2_metrics(bgr_, depth_, camera_model_, gt_xyz_):
    """Full pipeline check (Section 6, generalized): YOLO detect -> the
    node's exact backproject_pixel + radius correction -> base_link ->
    world, compared against ground truth."""
    result_ = model.predict(bgr_, conf=CONF_THRESHOLD, iou=0.45, imgsz=640, device="cpu", verbose=False)[0]
    best = None
    for box in result_.boxes:
        name_ = model.names.get(int(box.cls[0]), str(int(box.cls[0])))
        if TARGET_CLASSES and name_ not in TARGET_CLASSES:
            continue
        score_ = float(box.conf[0])
        if best is None or score_ > best[4]:
            x1_, y1_, x2_, y2_ = (float(v) for v in box.xyxy[0].tolist())
            best = (x1_, y1_, x2_, y2_, score_)

    if best is None:
        return dict(detected=False, backproject_ok=False)

    x1_, y1_, x2_, y2_, score_ = best
    u_ = 0.5 * (x1_ + x2_)
    v_ = 0.5 * (y1_ + y2_)
    r_px_ = 0.25 * ((x2_ - x1_) + (y2_ - y1_))

    pt_cam_ = backproject_pixel(u_, v_, depth_, camera_model_, window=2, min_valid_depth_m=0.05)
    if pt_cam_ is None:
        return dict(detected=True, backproject_ok=False, score=score_,
                    u=u_, v=v_, bbox=(x1_, y1_, x2_, y2_))

    fx_ = camera_model_.fx()
    z_ = pt_cam_.point.z
    pt_cam_.point.z += r_px_ * z_ / fx_

    p_opt_ = np.array([pt_cam_.point.x, pt_cam_.point.y, pt_cam_.point.z])
    p_base_ = apply_transform(BASE_T_CAM_OPTICAL, p_opt_)
    p_world_ = apply_transform(WORLD_T_BASE, p_base_)
    err_ = p_world_ - gt_xyz_

    return dict(detected=True, backproject_ok=True, score=score_, u=u_, v=v_,
                bbox=(x1_, y1_, x2_, y2_), p_world=p_world_, error=err_,
                error_norm=float(np.linalg.norm(err_)))


In [ ]:
POSITIONS = [("pos0 (original bag)", bgr, depth, camera_model, GT_SPHERE_XYZ)]
for _name in ["pos1", "pos2", "pos3", "pos4"]:
    _b, _d, _cm, _gt = load_lightweight_capture(NOTEBOOK_DIR / "perception_inputs_multi" / _name)
    POSITIONS.append((_name, _b, _d, _cm, _gt))

rows = []
overlays = []
for name, b, d, cm, gt in POSITIONS:
    t1 = test1_metrics(d, cm, gt)
    t2 = test2_metrics(b, d, cm, gt)
    rows.append({
        "position": name,
        "gt_world_xyz": np.round(gt, 3),
        "t1_inside_frame": t1["inside"],
        "t1_depth_residual_mm": round(t1["depth_residual"] * 1000, 1) if t1["inside"] else None,
        "t2_detected": t2["detected"],
        "t2_score": round(t2["score"], 2) if t2.get("score") is not None else None,
        "t2_error_cm": round(t2["error_norm"] * 100, 2) if t2.get("backproject_ok") else None,
    })
    overlays.append((name, b, t1, t2))

results_df = pd.DataFrame(rows)
results_df


In [ ]:
TOLERANCE_CM = TOLERANCE_M * 100
n = len(overlays)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
for ax, (name, b, t1, t2) in zip(axes, overlays):
    ax.imshow(b[:, :, ::-1])
    if t1["inside"]:
        ax.scatter([t1["u"]], [t1["v"]], c="red", marker="x", s=120, label="ground truth")
    if t2.get("backproject_ok"):
        x1_, y1_, x2_, y2_ = t2["bbox"]
        ax.add_patch(plt.Rectangle((x1_, y1_), x2_ - x1_, y2_ - y1_, fill=False, edgecolor="lime", linewidth=2))
        err_cm = t2["error_norm"] * 100
        tag = "PASS" if err_cm < TOLERANCE_CM else "FAIL"
        ax.set_title(f"{name}\n{tag} err={err_cm:.1f}cm", fontsize=9)
    else:
        ax.set_title(f"{name}\nno detection", fontsize=9, color="red")
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(loc="upper right", fontsize=7)
plt.tight_layout()
plt.show()

print(f"\n{results_df['t2_detected'].sum()}/{len(results_df)} detected; "
      f"{(results_df['t2_error_cm'] < TOLERANCE_CM).sum()}/{results_df['t2_error_cm'].notna().sum()} "
      f"of those within {TOLERANCE_CM:.0f} cm")


**Reading this table:** if all 5 rows pass, the transform chain and
detector hold up across the frame (near/far, left/right), not just at the
one spot the original bag happened to show — a meaningfully more
comprehensive claim than Section 5-6 alone. A position-dependent failure
pattern (e.g. only the two `pos3`/`pos4` "far" rows fail) would point at
something depth- or distortion-related rather than a wholesale transform
bug; a single outlier is more likely a YOLO miss on that particular render
than a geometry problem.

This still does **not** cover the moving-arm case — see Section 9.

## 9. Interpreting results / where to look next

- **Test 1 fails (predicted pixel off-image or large depth residual):**
  the static chain is wrong. Most likely causes, in order of likelihood:
  - The arm was **not** at the assumed sim-startup joint config when the
    bag was recorded (re-record with `/joint_states` or `/tf` to confirm).
  - A typo transcribing an xacro `xyz`/`rpy` into `URDF_CHAIN` above.
  - The robot was spawned with a different pose than
    `sim_robot.launch.py` currently specifies (check for launch overrides).

- **Test 1 passes, Test 2 fails:** the transform chain is fine; look at
  detection/back-projection instead:
  - Is YOLO actually finding the sphere (`candidates` empty or wrong box)?
    A synthetic render often looks nothing like COCO's "sports ball" —
    consider a fine-tuned `ball_yolo.pt`, or a simpler depth/color-blob
    detector for this controlled test.
  - Is the `min_valid_depth_m` / window logic pulling in background or
    gripper depth instead of the ball (see the long comment in
    `perception_geometry.backproject_pixel`)?
  - Timestamp sync: this bag is static, so `ApproximateTimeSynchronizer`
    slop shouldn't matter here, but would on a moving-arm bag.

- **Both pass, including Section 8's 5-position table:** the pipeline is
  validated for this static joint config across the visible workspace, not
  just at one spot. What's still unvalidated is the *moving*-arm case (real
  eye-in-hand operation) — for that, record a new bag that **includes `/tf`
  and `/joint_states`**, and drive `BASE_T_CAM_OPTICAL` from the recorded
  joint angles at each frame's timestamp instead of the fixed
  `DEFAULT_JOINT_POSITIONS` here. A handful of *static* holds at a few
  distinct `group_state`-style joint configs (each with its own
  lightweight capture, the way Section 8 did for ball position) would
  already catch a wrong sign/axis in `URDF_CHAIN`'s revolute joints without
  needing a full dynamic recording.